# 10 — Shared durable factual event contract

Configuration and tested offline helpers for the existing 11–14 + 99 sequence.
`logical_session_id != source_epoch`. Domains: FACTUAL_OBSERVATION, DERIVED_FACTUAL_METRIC, MODELLED_SCENARIO; the last is excluded from factual publication. Names/photos never enter Gold. Kart No is the external marketing join key. AIDP is outside the event-critical LIVE path.

Set `AIDP_PROJECT_ROOT` to the project checkout. Use one pipeline writer, a stable completed archive inventory, UTC, and the existing catalog/volumes. Imports reuse backend domain code without starting transport. The driver has explicit processing bounds and fails before downstream publication when exceeded.


In [ ]:
# Shared factual archive contract. Pure helpers are tested locally; Spark runs only in Workbench.
import os
import sys
import json
import hashlib
import math
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

CATALOG = "race_intelligence"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_EVENT_SCHEMA = "gold_event"
EVENT_CAPTURE_ROOT = "/Volumes/race_intelligence/bronze/event_capture"
EVENT_SESSION_ARCHIVE_ROOT = "/Volumes/race_intelligence/bronze/event_capture/sessions"
GOLD_EVENT_SERVING_ROOT = "/Volumes/race_intelligence/gold/serving/gold_event"
EVENT_CONTRACT_VERSION = "1.0"
# Bound driver memory and fail before partial publication. Raise deliberately for larger recordings.
MAX_RAW_FRAMES = int(os.getenv("AIDP_EVENT_MAX_RAW_FRAMES", "2000000"))
MAX_SILVER_ROWS = int(os.getenv("AIDP_EVENT_MAX_SILVER_ROWS", "3000000"))
EXPECTED_CHANNELS = set(filter(None, os.getenv("AIDP_EVENT_EXPECTED_CHANNELS",
    "timing.session.info,timing.session.status,timing.session.standings.overall.active,timing.analysis.laps").split(",")))
SUPPORTED_CHANNELS = {
    "gps", "rgps", "timing.session.info", "timing.session.entry", "timing.session.status",
    "timing.session.standings.overall.active", "timing.session.standings.overall.finishLine",
    "timing.session.standings.overall.participantDetails", "timing.analysis.laps",
    "timing.liveEvents.fl", "raceControl", "weather",
}
def project_root():
    configured = os.getenv("AIDP_PROJECT_ROOT")
    candidates = [Path(configured)] if configured else [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / "app/backend/app/integrations/aks/live_adapter.py").is_file():
            return path.resolve()
    raise RuntimeError("Set AIDP_PROJECT_ROOT to the project checkout in Workbench.")
PROJECT_ROOT = project_root()
sys.path.insert(0, str(PROJECT_ROOT / "app/backend"))
# Import the real domain implementation, never app.main or a network entrypoint.
from app.config import AksSettings
from app.integrations.aks.client import AksTimingClient
from app.integrations.aks.live_adapter import AksLiveIngestAdapter
from app.integrations.aks.live_events import InProcessEventBus, NormalizedEvent
from app.integrations.aks.protocol import parse_packet, parse_json_payload, CRLF
from app.services.baseball_card_evidence import CardEvidenceProjection
from app.services.event_archive import validate_completed_segment

def require(ok, message):
    if not ok:
        raise RuntimeError(message)

def utc(value):
    if isinstance(value, str):
        value = datetime.fromisoformat(value.replace("Z", "+00:00"))
    require(isinstance(value, datetime), "Missing/invalid evidence timestamp")
    return value.replace(tzinfo=timezone.utc) if value.tzinfo is None else value.astimezone(timezone.utc)

def stamp(value):
    return utc(value).isoformat(timespec="microseconds") if value is not None else None

def packed(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)

def digest(value):
    return hashlib.sha256(value if isinstance(value, bytes) else value.encode("utf-8")).hexdigest()

def signed(row):
    return {**row, "content_hash": digest(packed(row))}

def read_archive(root=EVENT_SESSION_ARCHIVE_ROOT):
    """Exact binary lines, hashes, metadata and safe path identity; no archive writes."""
    root = Path(root).resolve()
    require(root.is_dir(), "Durable event archive volume is unavailable")
    directories = sorted(p for p in root.glob("*/*") if p.is_dir())
    require(directories, "No completed archive segments: supply a real finalized segment")
    segments, frames = [], []
    for folder in directories:
        require(not folder.is_symlink(), "Archive segment symlinks are unsupported")
        source, files = validate_completed_segment(folder)
        body = (folder / "manifest.json").read_bytes()
        archive = json.loads((folder / "archive-manifest.json").read_bytes())
        require(archive.get("schema") == "race-intelligence.event-archive.v1" and
                archive.get("archiveStatus") == "ARCHIVED" and archive.get("source") == "AKS",
                "Invalid archive schema/status/source")
        session, segment = archive.get("logicalSessionId"), archive.get("segmentId")
        require(session == folder.parent.name == source["sessionId"] and segment == folder.name,
                "Archive path/session/segment conflict")
        require(archive.get("recorderStatus") == "RECORDER_FINISHED", "Unfinished archive")
        require(archive.get("recorderManifestSha256") == digest(body), "Manifest SHA-256 conflict")
        require(archive.get("files") == files, "Raw file hash/size manifest conflict")
        actual = {str(p.relative_to(folder)) for p in (folder / "raw").iterdir() if p.is_file()}
        require(actual == {f["path"] for f in files}, "Raw file inventory mismatch")
        recording = source["recording"]
        for field, expected in {
            "messageCount": recording["lineCount"], "jsonMessageCount": recording["jsonMessageCount"],
            "byteCount": recording["bytes"], "rawFileCount": len(files),
            "eventContractVersion": EVENT_CONTRACT_VERSION,
            "operationalContext": source.get("operationalContext"),
        }.items():
            require(archive.get(field) == expected, "Archive reconciliation failed: " + field)
        require(source.get("operationalContext") in {"ACTUAL_EVENT", "REHEARSAL"}, "Invalid factual context")
        require(utc(source["openedAt"]) <= utc(source["closedAt"]), "Invalid segment time range")
        segments.append(signed({
            "logical_session_id": session, "segment_id": segment,
            "operational_context": source["operationalContext"],
            "started_at": stamp(source["openedAt"]), "ended_at": stamp(source["closedAt"]),
            "message_count": recording["lineCount"], "json_message_count": recording["jsonMessageCount"],
            "byte_count": recording["bytes"], "raw_file_count": len(files),
            "recorder_status": "RECORDER_FINISHED", "archive_path": str(folder),
            "manifest_sha256": digest(body), "manifest_json": body.decode("utf-8"),
            "archive_json": packed(archive),
        }))
        for item in files:
            path = folder / item["path"]
            require(path.resolve().is_relative_to(root), "Raw path escapes archive volume")
            with path.open("rb") as handle:
                for original in handle:
                    require(original.endswith(b"\n"), "Truncated JSONL line")
                    text = original.decode("utf-8")
                    envelope = json.loads(text)
                    recorded = stamp(envelope["recordedAt"])
                    require(isinstance(envelope.get("line"), str), "Missing original protocol line")
                    frames.append({
                        "logical_session_id": session, "segment_id": segment,
                        "sequence": envelope["lineNumber"], "connection_id": envelope.get("connectionId"),
                        "recorded_at": recorded, "recorded_at_ms": envelope.get("recordedAtMs"),
                        "channel": envelope.get("channel"), "command": envelope.get("command"),
                        "message_id": envelope.get("messageId"),
                        "raw_payload": text, "raw_sha256": digest(original),
                        "raw_bytes": len(original), "source_file": item["path"],
                    })
                    require(len(frames) <= MAX_RAW_FRAMES, "Archive exceeds AIDP_EVENT_MAX_RAW_FRAMES; select a bounded volume scope")
    order = {(r["logical_session_id"], r["segment_id"]): r["started_at"] for r in segments}
    frames.sort(key=lambda r: (r["logical_session_id"], order[r["logical_session_id"], r["segment_id"]], r["segment_id"], r["sequence"]))
    return segments, frames

SEGMENT_DDL = """logical_session_id STRING, segment_id STRING, operational_context STRING,
started_at TIMESTAMP, ended_at TIMESTAMP, message_count BIGINT, json_message_count BIGINT,
byte_count BIGINT, raw_file_count BIGINT, recorder_status STRING, archive_path STRING,
manifest_sha256 STRING, manifest_json STRING, archive_json STRING, content_hash STRING, ingested_at TIMESTAMP"""
RAW_DDL = """logical_session_id STRING, segment_id STRING, sequence BIGINT, connection_id BIGINT,
recorded_at TIMESTAMP, recorded_at_ms BIGINT, channel STRING, command STRING, message_id STRING,
raw_payload STRING, raw_sha256 STRING, raw_bytes BIGINT, source_file STRING, ingested_at TIMESTAMP"""
EVENT_DDL = """logical_session_id STRING, source_epoch BIGINT, event_sequence BIGINT,
segment_id STRING, raw_sequence BIGINT, event_type STRING, channel STRING,
observed_at TIMESTAMP, recorded_at TIMESTAMP, operational_context STRING,
evidence_domain STRING, event_contract_version STRING, event_json STRING, content_hash STRING"""
ROW_DDL = EVENT_DDL + """, row_index BIGINT, participant_id STRING, kart_number STRING,
position BIGINT, lap_number BIGINT, gap_to_leader DOUBLE, interval DOUBLE,
lap_time_seconds DOUBLE, lap_valid BOOLEAN, latitude DOUBLE, longitude DOUBLE,
position_source STRING, source_entity_id STRING"""
CARD_DDL = """logical_session_id STRING, source_epoch BIGINT, participant_id STRING, kart_number STRING,
latest_observed_position BIGINT, peak_observed_position BIGINT, observed_lap BIGINT,
gap_to_leader DOUBLE, interval DOUBLE, best_valid_lap DOUBLE, start_position BIGINT,
final_position BIGINT, position_change BIGINT, updated_at TIMESTAMP,
operational_context STRING, evidence_domain STRING, event_contract_version STRING"""
LAP_DDL = """logical_session_id STRING, source_epoch BIGINT, participant_id STRING, kart_number STRING,
lap_number BIGINT, lap_time_seconds DOUBLE, lap_valid BOOLEAN, observed_at TIMESTAMP"""
SUMMARY_DDL = """logical_session_id STRING, source_epoch BIGINT, participant_count BIGINT,
event_count BIGINT, started_at TIMESTAMP, ended_at TIMESTAMP, updated_at TIMESTAMP,
session_status STRING, operational_context STRING, evidence_domain STRING, event_contract_version STRING"""
SILVER_TABLES = {
    "event_normalized_events": EVENT_DDL, "event_session": EVENT_DDL,
    "event_standings": ROW_DDL, "event_laps": ROW_DDL, "event_positions": ROW_DDL,
    "event_finish_line": ROW_DDL, "event_entries": ROW_DDL,
    "event_race_control": EVENT_DDL, "event_weather": EVENT_DDL, "event_live_events": EVENT_DDL,
}
GOLD_TABLES = {"participant_state": CARD_DDL, "baseball_card_evidence": CARD_DDL,
               "participant_laps": LAP_DDL, "session_summary": SUMMARY_DDL}

def table(schema, name):
    return f"{CATALOG}.{schema}.{name}"

def spark_frame(spark, rows, ddl):
    from pyspark.sql import functions as F, types as T
    schema = spark.createDataFrame([], ddl).schema
    # Explicitly convert ISO strings; never pass them as Python TimestampType values.
    input_schema = T.StructType([T.StructField(f.name, T.StringType() if isinstance(f.dataType, T.TimestampType) else f.dataType, True) for f in schema])
    rows = [{f.name: (stamp(row.get(f.name)) if isinstance(f.dataType, T.TimestampType) else row.get(f.name)) for f in schema} for row in rows]
    frame = spark.createDataFrame(rows, input_schema)
    for field in schema:
        if isinstance(field.dataType, T.TimestampType):
            invalid = frame.filter(F.col(field.name).isNotNull() & F.to_timestamp(field.name).isNull()).limit(1).count()
            require(not invalid, "Invalid timestamp: " + field.name)
            frame = frame.withColumn(field.name, F.to_timestamp(field.name))
    if "ingested_at" in frame.columns:
        frame = frame.withColumn("ingested_at", F.current_timestamp())
    return frame

def check_schema(spark, target, ddl, create=False):
    expected = spark.createDataFrame([], ddl).schema
    if not spark.catalog.tableExists(target):
        require(create, "Missing required table: " + target)
        spark.sql(f"CREATE TABLE {target} ({ddl}) USING DELTA")
    actual = spark.table(target).schema
    required = {f.name: f.dataType.simpleString() for f in expected}
    found = {f.name: f.dataType.simpleString() for f in actual}
    require(found == required, f"Incompatible schema {target}. Expected {required}; found {found}. Stop: approve a non-destructive migration/backfill before rerunning. No table was dropped.")

def preflight(spark, specs):
    # Inspect every existing table before creating/writing any target.
    for target, ddl in specs.items():
        if spark.catalog.tableExists(target):
            check_schema(spark, target, ddl)
    for target, ddl in specs.items():
        check_schema(spark, target, ddl, create=True)

def merge_rows(spark, target, rows, ddl, keys, immutable_hash=None):
    from pyspark.sql import functions as F
    if not rows:
        return
    incoming = spark_frame(spark, rows, ddl)
    require(not incoming.groupBy(*keys).count().filter("count > 1").limit(1).count(),
            "Duplicate incoming natural key: " + target)
    if immutable_hash:
        conflicts = incoming.alias("s").join(spark.table(target).alias("t"), keys).filter(
            ~F.col("s." + immutable_hash).eqNullSafe(F.col("t." + immutable_hash)))
        require(not conflicts.limit(1).count(), "CONFLICT: immutable evidence in " + target)
    incoming.createOrReplaceTempView("_durable_incoming")
    match = " AND ".join(f"t.{k}=s.{k}" for k in keys)
    update = "" if immutable_hash else "WHEN MATCHED THEN UPDATE SET * "
    spark.sql(f"MERGE INTO {target} t USING _durable_incoming s ON {match} {update}WHEN NOT MATCHED THEN INSERT *")

def bronze_input(spark):
    check_schema(spark, table(BRONZE_SCHEMA, "event_segments"), SEGMENT_DDL)
    check_schema(spark, table(BRONZE_SCHEMA, "event_raw_frames"), RAW_DDL)
    segments = [r.asDict() for r in spark.table(table(BRONZE_SCHEMA, "event_segments")).toLocalIterator()]
    require(segments, "Bronze contains no segments")
    count = spark.table(table(BRONZE_SCHEMA, "event_raw_frames")).count()
    require(0 < count <= MAX_RAW_FRAMES, "Bronze row count outside configured processing bound")
    frames = [r.asDict() for r in spark.table(table(BRONZE_SCHEMA, "event_raw_frames")).toLocalIterator()]
    order = {(r["logical_session_id"], r["segment_id"]): stamp(r["started_at"]) for r in segments}
    frames.sort(key=lambda r: (r["logical_session_id"], order[r["logical_session_id"], r["segment_id"]], r["segment_id"], r["sequence"]))
    return segments, frames

def normalize(segments, frames):
    """Offline replay through AksLiveIngestAdapter + SourceEpochs; no transport opened.
    SourceEpochs implements known info.id change, closed -> open, closed zero grid,
    coalescing, and no reconnect/lap-regression epoch; adapter clears old delta trees.
    """
    contexts = {r["logical_session_id"]: r["operational_context"] for r in segments}
    require(all(r["operational_context"] == contexts[r["logical_session_id"]] for r in segments), "Mixed operational contexts in one logical session")
    buses, adapters, events = {}, {}, []
    last_recorded = {}
    coverage = Counter()
    for frame in frames:
        session = frame["logical_session_id"]
        recorded = utc(frame["recorded_at"])
        require(session not in last_recorded or recorded >= last_recorded[session],
                "Overlapping/out-of-order segments: resolve archive ordering before normalization")
        last_recorded[session] = recorded
        if session not in adapters:
            buses[session] = InProcessEventBus()
            adapters[session] = AksLiveIngestAdapter(session_id=session,
                client=AksTimingClient(AksSettings(host=None)), publisher=buses[session],
                persist_raw=lambda line: None, operational_context=contexts[session])
        adapter, bus = adapters[session], buses[session]
        envelope = json.loads(frame["raw_payload"])
        require(digest(frame["raw_payload"]) == frame["raw_sha256"], "Bronze raw hash mismatch")
        wire = envelope["line"]
        packet = parse_packet(wire + ("" if wire.endswith(CRLF) else CRLF))
        if packet.command == "JSON":
            payload = parse_json_payload(packet)  # fail loudly on malformed JSON
            require(isinstance(payload, dict), "Unsupported JSON root")
            changed = adapter._changed_channels(payload, packet.channel)
            require(set(changed) <= SUPPORTED_CHANNELS, "Unsupported source channel")
            coverage.update(changed)
        adapter.connection_id = frame.get("connection_id") or 0
        before = adapter.sequence
        emitted = adapter.process_line(wire + ("" if wire.endswith(CRLF) else CRLF), recorded_at=utc(frame["recorded_at"]))
        # SourceEpochs tagged each event before ordered publication.
        require(not bus.consumer_errors and not bus.events_rejected, "Backend normalization failure")
        for event in emitted:
            data = event.to_dict()
            data["processing_times"] = {}
            data["source_process_id"] = None
            events.append(signed({
                "logical_session_id": session, "source_epoch": event.source_epoch,
                "event_sequence": event.sequence, "segment_id": frame["segment_id"],
                "raw_sequence": frame["sequence"], "event_type": event.event_type,
                "channel": event.source_channel, "observed_at": stamp(event.observed_at),
                "recorded_at": stamp(event.recorded_at), "operational_context": event.operational_context,
                "evidence_domain": event.evidence_domain, "event_contract_version": event.event_contract_version,
                "event_json": packed(data),
            }))
            require(len(events) <= MAX_SILVER_ROWS, "Normalized event processing bound exceeded")
    return events, dict(coverage)

def silver_rows(events):
    result = {name: [] for name in SILVER_TABLES}
    routing = {"SESSION_STATE": "event_session", "STANDINGS_UPDATE": "event_standings",
               "LAP_UPDATE": "event_laps", "POSITION_UPDATE": "event_positions",
               "FINISH_LINE_UPDATE": "event_finish_line", "PARTICIPANT_UPDATE": "event_entries",
               "RACE_CONTROL_UPDATE": "event_race_control", "WEATHER_UPDATE": "event_weather",
               "LIVE_EVENT_UPDATE": "event_live_events"}
    total = 0
    for event in events:
        result["event_normalized_events"].append(event)
        target = routing[event["event_type"]]
        if SILVER_TABLES[target] == EVENT_DDL:
            result[target].append(event)
            continue
        payload = json.loads(event["event_json"])["payload"]
        for index, row in enumerate(payload.get("rows", [])):
            item = {**event, "row_index": index, "participant_id": row.get("participantId"),
                "kart_number": row.get("externalParticipantNumber"), "position": row.get("position"),
                "lap_number": row.get("lapNumber"), "gap_to_leader": row.get("gapFirstSeconds"),
                "interval": row.get("gapPreviousSeconds"), "lap_time_seconds": row.get("lapTimeSeconds"),
                "lap_valid": row.get("lapValid"), "latitude": row.get("latitude"),
                "longitude": row.get("longitude"), "position_source": row.get("positionSource"),
                "source_entity_id": row.get("sourceEntityId")}
            for field in ("gap_to_leader", "interval", "lap_time_seconds", "latitude", "longitude"):
                if item[field] is not None:
                    item[field] = float(item[field])
                    require(math.isfinite(item[field]), "Non-finite factual metric")
            item.pop("content_hash")
            result[target].append(signed(item))
            total += 1
            require(total <= MAX_SILVER_ROWS, "Silver row processing bound exceeded")
    return result

def gold_rows(events):
    """Reconstruct each epoch separately using the exact backend card evidence projection."""
    projections, summaries, laps = {}, {}, {}
    for row in sorted(events, key=lambda r: (r["logical_session_id"], r["event_sequence"])):
        event = NormalizedEvent.from_dict(json.loads(row["event_json"]))
        require(event.evidence_domain == "FACTUAL_OBSERVATION", "Non-factual Silver input")
        key = (event.session_id, event.source_epoch)
        projection = projections.setdefault(key, CardEvidenceProjection(manage_source_boundaries=False))
        projection.publish(event)
        summary = summaries.setdefault(key, {
            "logical_session_id": event.session_id, "source_epoch": event.source_epoch,
            "participant_count": 0, "event_count": 0, "started_at": stamp(event.observed_at),
            "ended_at": None, "updated_at": stamp(event.observed_at), "session_status": "OBSERVED",
            "operational_context": event.operational_context, "evidence_domain": "FACTUAL_OBSERVATION",
            "event_contract_version": "1.0"})
        summary["event_count"] += 1
        summary["updated_at"] = stamp(event.observed_at)
        if event.source_channel == "timing.session.status":
            if event.payload.get("isClosed") is True:
                summary.update(session_status="CLOSED_OBSERVED", ended_at=stamp(event.observed_at))
            elif event.payload.get("isFinished") is True:
                summary.update(session_status="FINISHED_OBSERVED", ended_at=stamp(event.observed_at))
            elif event.payload.get("isSessionRunning") is True:
                summary["session_status"] = "RUNNING_OBSERVED"
        if event.event_type == "LAP_UPDATE":
            for lap in event.payload.get("rows", []):
                kart, number = lap.get("externalParticipantNumber"), lap.get("lapNumber")
                if not kart or kart.upper() == "SC" or not isinstance(number, int) or number <= 0:
                    continue
                laps[(*key, lap["participantId"], number)] = {
                    "logical_session_id": key[0], "source_epoch": key[1], "participant_id": lap["participantId"],
                    "kart_number": kart, "lap_number": number, "lap_time_seconds": lap.get("lapTimeSeconds"),
                    "lap_valid": lap.get("lapValid") is True, "observed_at": stamp(event.observed_at)}
    cards = []
    for (session, epoch), projection in projections.items():
        records = projection.result()
        summaries[session, epoch]["participant_count"] = len(records)
        for record in records.values():
            cards.append({
                "logical_session_id": session, "source_epoch": epoch, "participant_id": record["participantId"],
                "kart_number": str(record["kartNo"]), "latest_observed_position": record["latestObservedPosition"],
                "peak_observed_position": record["peakPosition"], "observed_lap": record.get("observedLap"),
                "gap_to_leader": record.get("gapToLeaderSeconds"), "interval": record.get("intervalSeconds"),
                "best_valid_lap": record.get("bestLapSeconds"), "start_position": record.get("startPosition"),
                "final_position": None, "position_change": None,
                "updated_at": summaries[session, epoch]["updated_at"],
                "operational_context": summaries[session, epoch]["operational_context"],
                "evidence_domain": "FACTUAL_OBSERVATION", "event_contract_version": "1.0"})
    return {"participant_state": cards, "baseball_card_evidence": cards,
            "participant_laps": list(laps.values()), "session_summary": list(summaries.values())}

def assert_private_free(columns):
    forbidden = {"name", "display_name", "participant_name", "photo_asset", "photo", "driver_name",
        "fastest_participant_name", "final_leader_name", "laps_when_max_passed", "time_spent_ahead_of_max",
        "field_performance_rank", "field_performance_display"}
    require(not forbidden.intersection(c.lower() for c in columns), "Identity/modelled serving column detected")

def read_events(spark):
    check_schema(spark, table(SILVER_SCHEMA, "event_normalized_events"), EVENT_DDL)
    df = spark.table(table(SILVER_SCHEMA, "event_normalized_events"))
    require(0 < df.count() <= MAX_SILVER_ROWS, "No normalized events or processing bound exceeded")
    return [r.asDict() for r in df.orderBy("logical_session_id", "event_sequence").toLocalIterator()]

def equal_rows(spark, target, expected, ddl, keys):
    check_schema(spark, target, ddl)
    actual = spark.table(target).drop("ingested_at")
    require(not actual.groupBy(*keys).count().filter("count > 1").limit(1).count(), "Duplicate keys: " + target)
    wanted = spark_frame(spark, expected, ddl).select(actual.columns)
    require(not actual.exceptAll(wanted).limit(1).count() and not wanted.exceptAll(actual).limit(1).count(),
            "Reconstruction/parity mismatch: " + target)

def validate_all(spark):
    """Mandatory gates; absent data/tables and parser errors fail, never silently skip."""
    from pyspark.sql import functions as F
    segments, raw = read_archive()
    equal_rows(spark, table(BRONZE_SCHEMA, "event_segments"), segments, SEGMENT_DDL, ["logical_session_id", "segment_id"])
    equal_rows(spark, table(BRONZE_SCHEMA, "event_raw_frames"), raw, RAW_DDL, ["logical_session_id", "segment_id", "sequence"])
    for segment in segments:
        records = [r for r in raw if (r["logical_session_id"], r["segment_id"]) == (segment["logical_session_id"], segment["segment_id"])]
        require(len(records) == segment["message_count"], "Message reconciliation failed")
        require(sum(r["command"] == "JSON" for r in records) == segment["json_message_count"], "JSON count reconciliation failed")
        require(sum(r["raw_bytes"] for r in records) == segment["byte_count"], "Byte reconciliation failed")
        require(len({r["source_file"] for r in records}) == segment["raw_file_count"], "File count reconciliation failed")
    events, coverage = normalize(segments, raw)
    for session in {r["logical_session_id"] for r in segments}:
        observed = {r["channel"] for r in events if r["logical_session_id"] == session}
        require(EXPECTED_CHANNELS <= observed, "Expected channel coverage missing for a logical session")
    require(events and EXPECTED_CHANNELS <= set(coverage), "No events or expected channel coverage missing")
    previous = {}
    for event in events:
        session, epoch = event["logical_session_id"], event["source_epoch"]
        prior = previous.get(session, 1)
        require(epoch in {prior, prior + 1}, "Non-contiguous source_epoch")
        previous[session] = epoch
        require(utc(event["observed_at"]) == utc(event["recorded_at"]), "Backend recorded-time evidence contract mismatch")
    for name, rows in silver_rows(events).items():
        keys = ["logical_session_id", "source_epoch", "event_sequence"] + (["row_index"] if SILVER_TABLES[name] == ROW_DDL else [])
        equal_rows(spark, table(SILVER_SCHEMA, name), rows, SILVER_TABLES[name], keys)
    reconstructed = gold_rows(events)
    for name, rows in reconstructed.items():
        keys = ["logical_session_id", "source_epoch"] + ([] if name == "session_summary" else ["participant_id"]) + (["lap_number"] if name == "participant_laps" else [])
        target = table(GOLD_EVENT_SCHEMA, name)
        equal_rows(spark, target, rows, GOLD_TABLES[name], keys)
        assert_private_free(spark.table(target).columns)
    cards = reconstructed["baseball_card_evidence"]
    for card in cards:
        require(card["kart_number"].upper() != "SC", "SC leaked into participant evidence")
        require(card["peak_observed_position"] <= card["latest_observed_position"], "Peak/Latest inconsistency")
        require(card["final_position"] is None and card["position_change"] is None, "Unsupported final classification")
    # Independent consumer API comparison on the same raw reconstruction, for every epoch.
    from app.services.live_baseball_cards import LiveBaseballCards
    backend = LiveBaseballCards()
    snapshots = {}
    for row in events:
        event = NormalizedEvent.from_dict(json.loads(row["event_json"]))
        backend.consume(event)
        snapshots[event.session_id, event.source_epoch] = backend.snapshot(event.session_id, "REPLAY")["rows"]
    mapping = {"latest_observed_position": "latestObservedPosition", "peak_observed_position": "Peak Position",
        "observed_lap": "observedLap", "gap_to_leader": "gapToLeaderSeconds", "interval": "intervalSeconds",
        "best_valid_lap": "bestLapSeconds", "start_position": "Start Position",
        "final_position": "Final Position", "position_change": "Position Change"}
    for card in cards:
        snapshot = snapshots[card["logical_session_id"], card["source_epoch"]]
        reference = next(r for r in snapshot if r["Participant ID"] == card["participant_id"])
        require(all(card[k] == reference[v] for k, v in mapping.items()), "Backend factual card parity failed")
    return {"segments": len(segments), "messages": len(raw), "files": sum(r["raw_file_count"] for r in segments),
        "bytes": sum(r["byte_count"] for r in segments), "json_messages": sum(r["json_message_count"] for r in segments),
        "parser_errors": 0, "duplicate_keys": 0, "conflicting_payloads": 0, "channels": coverage,
        "epochs": len(reconstructed["session_summary"]), "factual_cards": len(cards),
        "backend_parity": "PASS", "privacy": "PASS"}


## Workbench preflight

No catalogs, schemas or volumes are created here. Notebook 12 stops on legacy schema incompatibility and names the required migration. Execute definitions locally for tests; execute this preflight only in Workbench.


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
for schema in (BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_EVENT_SCHEMA):
    spark.sql(f"DESCRIBE SCHEMA {CATALOG}.{schema}").collect()
require(Path(EVENT_SESSION_ARCHIVE_ROOT).is_dir(), "Existing event_capture volume/archive path missing")
require(Path(GOLD_EVENT_SERVING_ROOT).parent.is_dir(), "Existing gold/serving volume missing")
print("Existing catalog, schemas and volume paths resolved; no data changed.")
